In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

print(f"Versión de pandas: {pd.__version__}")

Versión de pandas: 3.0.6


In [2]:
# Ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Carpetas principales
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"

print("Proyecto:", PROJECT_ROOT)
print("Datos originales:", DATA_RAW)

Proyecto: c:\Users\acoba\Documents\GitHub\Prueba_Analista_BI
Datos originales: c:\Users\acoba\Documents\GitHub\Prueba_Analista_BI\data\raw


In [3]:
archivos = sorted(DATA_RAW.iterdir())

print("Archivos encontrados:\n")

for archivo in archivos:
    print(f"- {archivo.name}")

Archivos encontrados:

- AFILIADOS_A_CARGO.xlsx
- BD_AFILIADOS.csv
- BD_CONSUMOS_PISCILAGO.csv


## 1. Carga y perfilamiento inicial de las fuentes

En esta etapa se inspeccionan las fuentes originales para identificar su estructura, formato y características generales antes de realizar transformaciones o cruces.

El objetivo es validar la correcta lectura de los archivos y reconocer posibles aspectos de calidad de datos que deban tratarse posteriormente.

In [4]:
# Inspección inicial de los archivos CSV
for nombre in ["BD_AFILIADOS.csv", "BD_CONSUMOS_PISCILAGO.csv"]:
    ruta = DATA_RAW / nombre

    print("=" * 80)
    print(nombre)
    print("=" * 80)

    with open(ruta, "r", encoding="utf-8-sig") as archivo:
        for _ in range(3):
            print(archivo.readline().strip())

    print()

BD_AFILIADOS.csv
;id_persona_Modificado;NumIdPersona_Modificado;Nombre;Categoria;Segmento_poblacional;Piramide1;Piramide2;RazonSocial;Genero
0;CC10818289;10818289;ORLANDO JOSEALGUERO SOSA;A;Joven;1 emp grandes;1.1 platinum;ALIADOS LABORALES S A S;M
1;CC10224342;10224342;CRISTIAN LEONARDOCORTES MENDEZ;B;Joven;1 emp grandes;1.1 platinum;TELEPERFORMANCE COLOMBIA S A S;M

BD_CONSUMOS_PISCILAGO.csv
PERIODO;ANIO;ESTADO;FECHA_COMPRA;DESC_PRODUCTO;VALOR_SUBTOTAL;SEGURO;IDENTIFICACION;NOMBRES;PUNTO_RECAUDO
202407;2024;FACTURADA;17/07/2024;ENTRADA PISCILAGO;60000.00;1;10000;C;TAQUILLA PISCILAGO
202402;2024;FACTURADA;24/02/2024;ENTRADA PISCILAGO;60000.00;0;10001;C;TAQUILLA PISCILAGO



### 1.1 Carga de las fuentes

A partir de la inspección de los archivos se identificó que las fuentes CSV utilizan punto y coma (`;`) como separador. 

Las bases se cargan inicialmente sin modificar su contenido, con el fin de conservar la estructura original y realizar posteriormente las validaciones de calidad correspondientes.

In [5]:
# Carga de las fuentes originales

df_afiliados = pd.read_csv(
    DATA_RAW / "BD_AFILIADOS.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

df_grupo_familiar = pd.read_excel(
    DATA_RAW / "AFILIADOS_A_CARGO.xlsx"
)

df_consumos = pd.read_csv(
    DATA_RAW / "BD_CONSUMOS_PISCILAGO.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print("Carga finalizada correctamente.")

Carga finalizada correctamente.


In [6]:
# Dimensiones de las fuentes

print(f"BD_AFILIADOS:           {df_afiliados.shape[0]:,} filas x {df_afiliados.shape[1]} columnas")
print(f"AFILIADOS_A_CARGO:      {df_grupo_familiar.shape[0]:,} filas x {df_grupo_familiar.shape[1]} columnas")
print(f"BD_CONSUMOS_PISCILAGO:  {df_consumos.shape[0]:,} filas x {df_consumos.shape[1]} columnas")

BD_AFILIADOS:           448,951 filas x 10 columnas
AFILIADOS_A_CARGO:      39,733 filas x 2 columnas
BD_CONSUMOS_PISCILAGO:  145,060 filas x 10 columnas


In [7]:
# Nombres de las variables

for nombre, df in {
    "BD_AFILIADOS": df_afiliados,
    "AFILIADOS_A_CARGO": df_grupo_familiar,
    "BD_CONSUMOS_PISCILAGO": df_consumos
}.items():

    print("\n" + "=" * 70)
    print(nombre)
    print("=" * 70)

    for columna in df.columns:
        print(repr(columna))


BD_AFILIADOS
'Unnamed: 0'
'id_persona_Modificado'
'NumIdPersona_Modificado'
'Nombre'
'Categoria'
'Segmento_poblacional'
'Piramide1'
'Piramide2'
'RazonSocial'
'Genero'

AFILIADOS_A_CARGO
'NUM_IDENTIFICACION_AFILIADO'
'NUM_IDENTIFICACION_PERSONA_A_CARGO'

BD_CONSUMOS_PISCILAGO
'PERIODO'
'ANIO'
'ESTADO'
'FECHA_COMPRA'
'DESC_PRODUCTO'
'VALOR_SUBTOTAL'
'SEGURO'
'IDENTIFICACION'
'NOMBRES'
'PUNTO_RECAUDO'


In [8]:
display(df_afiliados.head())

,Unnamed: 0,id_persona_Modificado,NumIdPersona_Modificado,Nombre,Categoria,Segmento_poblacional,Piramide1,Piramide2,RazonSocial,Genero
0,0,CC10818289,10818289,ORLANDO JOSEALGUERO SOSA,A,Joven,1 emp grandes,1.1 platinum,ALIADOS LABORALES S A S,M
1,1,CC10224342,10224342,CRISTIAN LEONARDOCORTES MENDEZ,B,Joven,1 emp grandes,1.1 platinum,TELEPERFORMANCE COLOMBIA S A S,M
2,2,CC10814050,10814050,LAURA NIKOLERAMIREZ LOZANO,A,Basico,4 micro,4.5 transaccional,FABIO ORLANDO CASTIBLANCO CALIXTO,F
3,3,CC31064,31064,JOSE EDILBERTOBOHORQUEZ BEJARANO,A,Medio,4 micro,4.5 transaccional,ASOCIACION DE JUNTAS DE ACCION COMUNAL D EL MU...,M
4,4,CC10957942,10957942,JOSEFINA ANGARITA PINZON,A,Basico,4 micro,4.5 transaccional,PAPA PACHO LECHERIA SAS,F


In [9]:
display(df_grupo_familiar.head())

,NUM_IDENTIFICACION_AFILIADO,NUM_IDENTIFICACION_PERSONA_A_CARGO
0,1775,10036404
1,3824,96022201
2,4133,99022409
3,3084,10003904
4,3859,10007324


In [10]:
display(df_consumos.head())

,PERIODO,ANIO,ESTADO,FECHA_COMPRA,DESC_PRODUCTO,VALOR_SUBTOTAL,SEGURO,IDENTIFICACION,NOMBRES,PUNTO_RECAUDO
0,202407,2024,FACTURADA,17/07/2024,ENTRADA PISCILAGO,60000.0,1,10000,C,TAQUILLA PISCILAGO
1,202402,2024,FACTURADA,24/02/2024,ENTRADA PISCILAGO,60000.0,0,10001,C,TAQUILLA PISCILAGO
2,202401,2024,FACTURADA,1/01/2024,ENTRADA PISCILAGO,53000.0,0,10002,IN,TAQUILLA PISCILAGO
3,202401,2024,FACTURADA,27/01/2024,PISCITOUR,93333.0,0,10003,C,TAQUILLA PISCILAGO
4,202401,2024,FACTURADA,10/01/2024,ENTRADA PISCILAGO,60000.0,0,10004,C,TAQUILLA PISCILAGO


### 1.2 Perfilamiento de calidad de datos

Se realiza un diagnóstico inicial de las fuentes para identificar tipos de datos, valores nulos, registros duplicados y posibles llaves de integración.

En esta etapa no se modifican los datos. Los hallazgos permitirán definir posteriormente las reglas de limpieza y transformación.

In [11]:
# Resumen general de calidad de las fuentes

fuentes = {
    "BD_AFILIADOS": df_afiliados,
    "AFILIADOS_A_CARGO": df_grupo_familiar,
    "BD_CONSUMOS_PISCILAGO": df_consumos
}

resumen = []

for nombre, df in fuentes.items():
    resumen.append({
        "fuente": nombre,
        "filas": len(df),
        "columnas": len(df.columns),
        "nulos_totales": int(df.isna().sum().sum()),
        "filas_duplicadas": int(df.duplicated().sum())
    })

df_resumen = pd.DataFrame(resumen)

display(df_resumen)

,fuente,filas,columnas,nulos_totales,filas_duplicadas
0,BD_AFILIADOS,448951,10,0,0
1,AFILIADOS_A_CARGO,39733,2,0,0
2,BD_CONSUMOS_PISCILAGO,145060,10,2797,0


In [12]:
def perfil_columnas(df):
    perfil = pd.DataFrame({
        "tipo_dato": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "porcentaje_nulos": (df.isna().mean() * 100).round(2),
        "valores_unicos": df.nunique(dropna=True)
    })

    return perfil


for nombre, df in fuentes.items():
    print(f"\n{nombre}")
    display(perfil_columnas(df))


BD_AFILIADOS


,tipo_dato,nulos,porcentaje_nulos,valores_unicos
Unnamed: 0,int64,0,0.0,448951
id_persona_Modificado,str,0,0.0,305637
NumIdPersona_Modificado,str,0,0.0,303956
Nombre,str,0,0.0,447718
Categoria,str,0,0.0,3
Segmento_poblacional,str,0,0.0,4
Piramide1,str,0,0.0,5
Piramide2,str,0,0.0,15
RazonSocial,str,0,0.0,54831
Genero,str,0,0.0,2



AFILIADOS_A_CARGO


,tipo_dato,nulos,porcentaje_nulos,valores_unicos
NUM_IDENTIFICACION_AFILIADO,int64,0,0.0,29879
NUM_IDENTIFICACION_PERSONA_A_CARGO,object,0,0.0,24555



BD_CONSUMOS_PISCILAGO


,tipo_dato,nulos,porcentaje_nulos,valores_unicos
PERIODO,int64,0,0.00,9
ANIO,int64,0,0.00,1
ESTADO,str,0,0.00,2
FECHA_COMPRA,str,0,0.00,259
DESC_PRODUCTO,str,0,0.00,67
VALOR_SUBTOTAL,float64,0,0.00,110
SEGURO,int64,0,0.00,2
IDENTIFICACION,int64,0,0.00,145060
NOMBRES,str,2797,1.93,78882
PUNTO_RECAUDO,str,0,0.00,37


In [13]:
llaves = {
    "BD_AFILIADOS": "NumIdPersona_Modificado",
    "AFILIADOS_A_CARGO": "NUM_IDENTIFICACION_PERSONA_A_CARGO",
    "BD_CONSUMOS_PISCILAGO": "IDENTIFICACION"
}

resultados_llaves = []

for nombre, columna in llaves.items():
    df = fuentes[nombre]

    resultados_llaves.append({
        "fuente": nombre,
        "variable": columna,
        "registros": len(df),
        "no_nulos": df[columna].notna().sum(),
        "valores_unicos": df[columna].nunique(dropna=True),
        "duplicados": df[columna].duplicated().sum()
    })

df_llaves = pd.DataFrame(resultados_llaves)

display(df_llaves)

,fuente,variable,registros,no_nulos,valores_unicos,duplicados
0,BD_AFILIADOS,NumIdPersona_Modificado,448951,448951,303956,144995
1,AFILIADOS_A_CARGO,NUM_IDENTIFICACION_PERSONA_A_CARGO,39733,39733,24555,15178
2,BD_CONSUMOS_PISCILAGO,IDENTIFICACION,145060,145060,145060,0


In [14]:
indice_exportado = df_afiliados["Unnamed: 0"]

print("Registros:", len(indice_exportado))
print("Valores únicos:", indice_exportado.nunique())
print("Nulos:", indice_exportado.isna().sum())
print("Mínimo:", indice_exportado.min())
print("Máximo:", indice_exportado.max())

print(
    "¿Coincide exactamente con un índice consecutivo?:",
    indice_exportado.tolist() == list(range(len(df_afiliados)))
)

Registros: 448951
Valores únicos: 448951
Nulos: 0
Mínimo: 0
Máximo: 448950
¿Coincide exactamente con un índice consecutivo?: True


In [15]:
id_afiliado = "NumIdPersona_Modificado"

total_registros = len(df_afiliados)
personas_unicas = df_afiliados[id_afiliado].nunique(dropna=True)

print(f"Registros en BD_AFILIADOS: {total_registros:,}")
print(f"Identificaciones únicas:   {personas_unicas:,}")
print(f"Diferencia:                {total_registros - personas_unicas:,}")

Registros en BD_AFILIADOS: 448,951
Identificaciones únicas:   303,956
Diferencia:                144,995


In [16]:
frecuencia_afiliados = (
    df_afiliados[id_afiliado]
    .value_counts()
    .rename_axis("identificacion")
    .reset_index(name="numero_registros")
)

display(frecuencia_afiliados.head(10))

,identificacion,numero_registros
0,10224342,2
1,10814050,2
2,31064,2
3,170903,2
4,10223482,2
5,2083,2
6,10575466,2
7,10003304,2
8,10161093,2
9,527708,2


### 1.3 Análisis de duplicidad en afiliados

La identificación del afiliado no es única en la fuente `BD_AFILIADOS`. Aunque no existen filas exactamente duplicadas, una misma identificación puede estar presente en más de un registro.

Antes de construir una dimensión única de afiliados se analizan las variables en las que difieren estos registros, con el fin de evitar una deduplicación arbitraria que pueda ocasionar pérdida de información.

In [17]:
# Identificaciones que aparecen más de una vez

ids_repetidos = (
    frecuencia_afiliados
    .query("numero_registros > 1")
    ["identificacion"]
)

afiliados_repetidos = (
    df_afiliados[
        df_afiliados[id_afiliado].isin(ids_repetidos)
    ]
    .sort_values(id_afiliado)
)

print(f"Identificaciones repetidas: {len(ids_repetidos):,}")
print(f"Registros involucrados:     {len(afiliados_repetidos):,}")

Identificaciones repetidas: 144,995
Registros involucrados:     289,990


In [18]:
columnas_analisis = [
    "NumIdPersona_Modificado",
    "id_persona_Modificado",
    "Nombre",
    "Categoria",
    "Segmento_poblacional",
    "Piramide1",
    "Piramide2",
    "RazonSocial",
    "Genero"
]

display(
    afiliados_repetidos[columnas_analisis].head(20)
)

,NumIdPersona_Modificado,id_persona_Modificado,Nombre,Categoria,Segmento_poblacional,Piramide1,Piramide2,RazonSocial,Genero
334461,10000,CE10000,ANALICIA PEREZ MARQUEZ,B,Medio,1 emp grandes,1.1 platinum,TELEPERFORMANCE COLOMBIA S A S,F
188604,10000,PT10000,GABRIELA CAROLINALOPEZ RODRIGUEZ,A,Basico,4 micro,4.1 estandar,MULTI IMPRESOS S A S,F
48512,10000002,CC10000002,DIANA MARISELAMAYORGA LANCHEROS,A,Basico,3 empresas pymes,3.2 vip estandar,UNIVERSAL DE LIMPIEZA S A S,F
408913,10000002,CC10000002,DANNA LISBETHCARDOZO ANDRADE,A,Basico,3 empresas pymes,3.1 vip,CEKAED SECURITY LTDA,F
193828,10000003,CC10000003,JEFERSON ANDRESVARGAS PIMENTEL,A,Joven,3 empresas pymes,3.2 vip estandar,SEGURIDAD SPRINT LTDA,M
321285,10000003,CC10000003,JENIFER LISETHRAMIREZ GALICIA,A,Basico,4 micro,4.5 transaccional,AUDITORIAS CONSULTORIAS E INGENIERIA AUDITO S A S,F
313237,10000004,CC10000004,KAREN TERESATORRES CARDONA,A,Basico,1 emp grandes,1.1 platinum,GENTE OPORTUNA S A S,F
84737,10000004,CC10000004,JHULIANA ANDREAROCHA RUBIANO,A,Basico,1 emp grandes,1.1 platinum,MEDICALL TALENTO HUMANO S A S,F
114934,10000005,CC10000005,ESTEFANIA ALVAREZ ZAPATA,A,Basico,2 emp medio,2.2 silver,QUICK BPO SAS,F
21075,10000005,CC10000005,EMERSON ESNEIDERLEIVA ROJAS,A,Joven,3 empresas pymes,3.1 vip,MUEVE USME SAS,M


In [19]:
variables_afiliado = [
    "id_persona_Modificado",
    "Nombre",
    "Categoria",
    "Segmento_poblacional",
    "Piramide1",
    "Piramide2",
    "RazonSocial",
    "Genero"
]

variabilidad = {}

for columna in variables_afiliado:
    cantidad_valores = (
        df_afiliados
        .groupby("NumIdPersona_Modificado")[columna]
        .nunique(dropna=False)
    )

    variabilidad[columna] = (cantidad_valores > 1).sum()

variabilidad_df = (
    pd.DataFrame.from_dict(
        variabilidad,
        orient="index",
        columns=["personas_con_mas_de_un_valor"]
    )
    .sort_values(
        "personas_con_mas_de_un_valor",
        ascending=False
    )
)

display(variabilidad_df)

,personas_con_mas_de_un_valor
Nombre,144983
RazonSocial,143751
Piramide2,123398
Piramide1,97634
Segmento_poblacional,76230
Categoria,48607
Genero,40235
id_persona_Modificado,1681


In [20]:
distribucion_frecuencia = (
    frecuencia_afiliados["numero_registros"]
    .value_counts()
    .sort_index()
    .rename_axis("numero_registros_por_persona")
    .reset_index(name="cantidad_personas")
)

display(distribucion_frecuencia)

,numero_registros_por_persona,cantidad_personas
0,1,158961
1,2,144995


In [21]:
relaciones_persona_cargo = (
    df_grupo_familiar
    .groupby("NUM_IDENTIFICACION_PERSONA_A_CARGO")
    ["NUM_IDENTIFICACION_AFILIADO"]
    .nunique()
)

resumen_relaciones = (
    relaciones_persona_cargo
    .value_counts()
    .sort_index()
    .rename_axis("cantidad_afiliados_asociados")
    .reset_index(name="personas_a_cargo")
)

display(resumen_relaciones)

,cantidad_afiliados_asociados,personas_a_cargo
0,1,16699
1,2,4129
2,3,1865
3,4,951
4,5,466
5,6,228
6,7,120
7,8,64
8,9,17
9,10,9


In [22]:
print(
    "Personas a cargo asociadas a más de un afiliado:",
    (relaciones_persona_cargo > 1).sum()
)

Personas a cargo asociadas a más de un afiliado: 7856


### 1.4 Evaluación de la llave de identificación del afiliado

El campo `NumIdPersona_Modificado` presenta colisiones entre personas diferentes debido a que conserva únicamente el número de documento y no el tipo de identificación.

Por esta razón, se evalúa `id_persona_Modificado`, que combina tipo y número de documento, como posible identificador único del afiliado.

In [23]:
id_completo = "id_persona_Modificado"

print(f"Registros totales:          {len(df_afiliados):,}")
print(f"IDs completos únicos:       {df_afiliados[id_completo].nunique():,}")
print(f"IDs completos repetidos:    {df_afiliados[id_completo].duplicated().sum():,}")

Registros totales:          448,951
IDs completos únicos:       305,637
IDs completos repetidos:    143,314


In [24]:
frecuencia_id_completo = (
    df_afiliados[id_completo]
    .value_counts()
    .rename_axis("id_persona")
    .reset_index(name="numero_registros")
)

display(
    frecuencia_id_completo
    .query("numero_registros > 1")
    .head(20)
)

,id_persona,numero_registros
0,CC10224342,2
1,CC10814050,2
2,CC31064,2
3,CC170903,2
4,CC10223482,2
5,CC2083,2
6,CC10575466,2
7,CC10003304,2
8,CC10161093,2
9,CC527708,2


In [25]:
distribucion_id_completo = (
    frecuencia_id_completo["numero_registros"]
    .value_counts()
    .sort_index()
    .rename_axis("numero_registros_por_id")
    .reset_index(name="cantidad_ids")
)

display(distribucion_id_completo)

,numero_registros_por_id,cantidad_ids
0,1,162323
1,2,143314


In [26]:
print(df_consumos["IDENTIFICACION"].head(20).tolist())

[10000, 10001, 10002, 10003, 10004, 10005, 10006, 10010, 10011, 10013, 10014, 10015, 10019, 10022, 10026, 10028, 10031, 10032, 10033, 10044]


In [27]:
print("Mínimo:", df_consumos["IDENTIFICACION"].min())
print("Máximo:", df_consumos["IDENTIFICACION"].max())

Mínimo: 10000
Máximo: 98686856


### 1.5 Validación de duplicados de negocio

La fuente contiene una columna técnica (`Unnamed: 0`) correspondiente al índice de exportación. 
Al ser única para cada fila, esta variable impide detectar duplicados exactos mediante una comparación directa de registros.

Por esta razón, la duplicidad se reevalúa excluyendo dicho campo técnico.

In [28]:
# Columnas de negocio, excluyendo el índice técnico de exportación

columnas_negocio = [
    columna
    for columna in df_afiliados.columns
    if columna != "Unnamed: 0"
]

duplicados_negocio = df_afiliados.duplicated(
    subset=columnas_negocio,
    keep=False
)

print(
    f"Filas involucradas en duplicados exactos de negocio: "
    f"{duplicados_negocio.sum():,}"
)

print(
    f"Duplicados adicionales que podrían eliminarse: "
    f"{df_afiliados.duplicated(subset=columnas_negocio).sum():,}"
)

Filas involucradas en duplicados exactos de negocio: 0
Duplicados adicionales que podrían eliminarse: 0


In [29]:
ids_completos_repetidos = (
    frecuencia_id_completo
    .query("numero_registros > 1")
    ["id_persona"]
)

muestra_ids_repetidos = (
    df_afiliados[
        df_afiliados["id_persona_Modificado"]
        .isin(ids_completos_repetidos.head(10))
    ]
    .sort_values("id_persona_Modificado")
)

display(
    muestra_ids_repetidos[
        [
            "id_persona_Modificado",
            "NumIdPersona_Modificado",
            "Nombre",
            "Categoria",
            "Segmento_poblacional",
            "Piramide1",
            "Piramide2",
            "RazonSocial",
            "Genero"
        ]
    ]
)

,id_persona_Modificado,NumIdPersona_Modificado,Nombre,Categoria,Segmento_poblacional,Piramide1,Piramide2,RazonSocial,Genero
13,CC10003304,10003304,NATALIA BERNAL BORJA,A,Basico,3 empresas pymes,3.1 vip,ASIGNAR S A S,F
111738,CC10003304,10003304,DANIELA ARIAS PENAGOS,A,Joven,4 micro,4.1 estandar,TECSER LABORATORIOS S A,F
311945,CC10161093,10161093,ANA MARIAQUIMBAYO SUAREZ,A,Joven,3 empresas pymes,3.1 vip,DIEBOLD NIXDORF COLOMBIA SAS,F
14,CC10161093,10161093,SANTIAGO ANDRESMONROY MARTINEZ,A,Basico,2 emp medio,2.1 gold,UNIVERSIDAD MANUELA BELTRAN UMB,M
276726,CC10223482,10223482,GUSTAVO ADOLFODUARTE CORTES,A,Joven,1 emp grandes,1.2 premium,CONSTRUCTORA CONCONCRETO S A,M
9,CC10223482,10223482,JOHN ALEXANDERRODRIGUEZ AROCA,A,Basico,3 empresas pymes,3.2 vip estandar,CELAR LIMITADA,M
1,CC10224342,10224342,CRISTIAN LEONARDOCORTES MENDEZ,B,Joven,1 emp grandes,1.1 platinum,TELEPERFORMANCE COLOMBIA S A S,M
276008,CC10224342,10224342,JULIETH JIMENAMARTINEZ ALFONSO,A,Joven,4 micro,4.5 transaccional,COLENVAL SAS,F
127009,CC10575466,10575466,PILAR ALBARRACIN SANDOVAL,A,Basico,2 emp medio,2.1 gold,COLVISEG LIMITADA,F
12,CC10575466,10575466,EDGAR FERNEYBARON BLANCO,A,Basico,4 micro,4.5 transaccional,HANDITECH S A S,M


In [30]:
variables_comparacion = [
    "NumIdPersona_Modificado",
    "Nombre",
    "Categoria",
    "Segmento_poblacional",
    "Piramide1",
    "Piramide2",
    "RazonSocial",
    "Genero"
]

variabilidad_id_completo = {}

for columna in variables_comparacion:

    n_valores = (
        df_afiliados[
            df_afiliados["id_persona_Modificado"]
            .isin(ids_completos_repetidos)
        ]
        .groupby("id_persona_Modificado")[columna]
        .nunique(dropna=False)
    )

    variabilidad_id_completo[columna] = (n_valores > 1).sum()

variabilidad_id_completo_df = (
    pd.DataFrame.from_dict(
        variabilidad_id_completo,
        orient="index",
        columns=["ids_con_mas_de_un_valor"]
    )
    .sort_values("ids_con_mas_de_un_valor", ascending=False)
)

display(variabilidad_id_completo_df)

,ids_con_mas_de_un_valor
Nombre,143305
RazonSocial,142081
Piramide2,121990
Piramide1,96588
Segmento_poblacional,75221
Categoria,48005
Genero,39503
NumIdPersona_Modificado,0


### 1.6 Evaluación de la llave de integración con consumos

Dado que los identificadores de la base de afiliados no son únicos, antes de realizar el cruce se evalúa la cardinalidad de la relación entre las identificaciones registradas en las compras de Piscilago y la base de afiliados.

El propósito es identificar cuántos compradores presentan:

- ninguna coincidencia,
- una única coincidencia,
- más de una coincidencia.

Esta validación permite evitar relaciones muchos-a-muchos que puedan duplicar transacciones y distorsionar los indicadores.

In [31]:
# Cantidad de registros de afiliados asociados a cada número de identificación

conteo_afiliados_por_id = (
    df_afiliados
    .groupby("NumIdPersona_Modificado")
    .size()
)

# Identificaciones presentes en consumos

ids_consumos = df_consumos["IDENTIFICACION"]

# Cantidad de coincidencias de cada comprador en BD_AFILIADOS

coincidencias_afiliados = (
    ids_consumos
    .map(conteo_afiliados_por_id)
    .fillna(0)
    .astype(int)
)

resumen_coincidencias = (
    coincidencias_afiliados
    .value_counts()
    .sort_index()
    .rename_axis("numero_coincidencias")
    .reset_index(name="compradores")
)

resumen_coincidencias["porcentaje"] = (
    resumen_coincidencias["compradores"]
    / len(df_consumos)
    * 100
).round(2)

display(resumen_coincidencias)

,numero_coincidencias,compradores,porcentaje
0,0,145060,100.0


In [32]:
ids_consumo_ambiguos = ids_consumos[
    coincidencias_afiliados > 1
].unique()

print(
    f"Compradores con más de una coincidencia en afiliados: "
    f"{len(ids_consumo_ambiguos):,}"
)

Compradores con más de una coincidencia en afiliados: 0


In [33]:
afiliados_ambiguos_consumo = (
    df_afiliados[
        df_afiliados["NumIdPersona_Modificado"]
        .isin(ids_consumo_ambiguos)
    ]
    .sort_values("NumIdPersona_Modificado")
)

display(
    afiliados_ambiguos_consumo[
        [
            "NumIdPersona_Modificado",
            "id_persona_Modificado",
            "Nombre",
            "Categoria",
            "Segmento_poblacional",
            "Piramide1",
            "Piramide2",
            "RazonSocial",
            "Genero"
        ]
    ].head(20)
)

,NumIdPersona_Modificado,id_persona_Modificado,Nombre,Categoria,Segmento_poblacional,Piramide1,Piramide2,RazonSocial,Genero


In [34]:
dimensiones_bi = [
    "Categoria",
    "Segmento_poblacional",
    "Piramide1",
    "Piramide2"
]

ambiguedad_dimensiones = []

for columna in dimensiones_bi:

    conteo = (
        afiliados_ambiguos_consumo
        .groupby("NumIdPersona_Modificado")[columna]
        .nunique(dropna=False)
    )

    ambiguedad_dimensiones.append({
        "dimension": columna,
        "compradores_ambiguos": int((conteo > 1).sum())
    })

df_ambiguedad = pd.DataFrame(ambiguedad_dimensiones)

display(df_ambiguedad)

,dimension,compradores_ambiguos
0,Categoria,0
1,Segmento_poblacional,0
2,Piramide1,0
3,Piramide2,0


In [35]:
ids_no_en_afiliados = ids_consumos[
    coincidencias_afiliados == 0
]

print(
    f"Compradores sin coincidencia directa en afiliados: "
    f"{len(ids_no_en_afiliados):,}"
)

Compradores sin coincidencia directa en afiliados: 145,060


In [36]:
persona_cargo_normalizada = (
    df_grupo_familiar["NUM_IDENTIFICACION_PERSONA_A_CARGO"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

ids_no_afiliados_texto = (
    ids_no_en_afiliados
    .astype(str)
    .str.strip()
)

In [37]:
conteo_persona_cargo = (
    persona_cargo_normalizada
    .value_counts()
)

coincidencias_cargo = (
    ids_no_afiliados_texto
    .map(conteo_persona_cargo)
    .fillna(0)
    .astype(int)
)

resumen_cargo = (
    coincidencias_cargo
    .value_counts()
    .sort_index()
    .rename_axis("numero_coincidencias")
    .reset_index(name="compradores")
)

resumen_cargo["porcentaje"] = (
    resumen_cargo["compradores"]
    / len(ids_no_afiliados_texto)
    * 100
).round(2)

display(resumen_cargo)

,numero_coincidencias,compradores,porcentaje
0,0,129756,89.45
1,1,8509,5.87
2,2,3382,2.33
3,3,1718,1.18
4,4,858,0.59
5,5,427,0.29
6,6,208,0.14
7,7,114,0.08
8,8,60,0.04
9,9,14,0.01


### 1.7 Normalización de las llaves de integración

Las fuentes almacenan los números de identificación con tipos de datos diferentes. En `BD_AFILIADOS` el número de documento se encuentra como texto, mientras que en `BD_CONSUMOS_PISCILAGO` se encuentra como entero y en `AFILIADOS_A_CARGO` presenta un tipo mixto.

Antes de realizar los cruces, se homologan las llaves a formato texto. Las variables originales se conservan sin modificación para mantener la trazabilidad de las fuentes.

In [38]:
afiliados = df_afiliados.copy()
grupo_familiar = df_grupo_familiar.copy()
consumos = df_consumos.copy()

In [39]:
# Normalización de identificadores para integración

afiliados["id_normalizado"] = (
    afiliados["NumIdPersona_Modificado"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

grupo_familiar["id_afiliado_normalizado"] = (
    grupo_familiar["NUM_IDENTIFICACION_AFILIADO"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

grupo_familiar["id_persona_cargo_normalizado"] = (
    grupo_familiar["NUM_IDENTIFICACION_PERSONA_A_CARGO"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

consumos["id_normalizado"] = (
    consumos["IDENTIFICACION"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

In [40]:
print("AFILIADOS")
display(
    afiliados[
        ["NumIdPersona_Modificado", "id_normalizado"]
    ].head()
)

print("CONSUMOS")
display(
    consumos[
        ["IDENTIFICACION", "id_normalizado"]
    ].head()
)

print("PERSONAS A CARGO")
display(
    grupo_familiar[
        [
            "NUM_IDENTIFICACION_PERSONA_A_CARGO",
            "id_persona_cargo_normalizado"
        ]
    ].head()
)

AFILIADOS


,NumIdPersona_Modificado,id_normalizado
0,10818289,10818289
1,10224342,10224342
2,10814050,10814050
3,31064,31064
4,10957942,10957942


CONSUMOS


,IDENTIFICACION,id_normalizado
0,10000,10000
1,10001,10001
2,10002,10002
3,10003,10003
4,10004,10004


PERSONAS A CARGO


,NUM_IDENTIFICACION_PERSONA_A_CARGO,id_persona_cargo_normalizado
0,10036404,10036404
1,96022201,96022201
2,99022409,99022409
3,10003904,10003904
4,10007324,10007324


In [41]:
print(afiliados["id_normalizado"].dtype)
print(consumos["id_normalizado"].dtype)
print(grupo_familiar["id_persona_cargo_normalizado"].dtype)

string
string
string


In [42]:
conteo_afiliados_por_id = (
    afiliados
    .groupby("id_normalizado")
    .size()
)

coincidencias_afiliados = (
    consumos["id_normalizado"]
    .map(conteo_afiliados_por_id)
    .fillna(0)
    .astype(int)
)

resumen_coincidencias = (
    coincidencias_afiliados
    .value_counts()
    .sort_index()
    .rename_axis("numero_coincidencias")
    .reset_index(name="compradores")
)

resumen_coincidencias["porcentaje"] = (
    resumen_coincidencias["compradores"]
    / len(consumos)
    * 100
).round(2)

display(resumen_coincidencias)

,numero_coincidencias,compradores,porcentaje
0,0,48304,33.3
1,1,19432,13.4
2,2,77324,53.3


In [43]:
print(
    "Sin coincidencia:",
    (coincidencias_afiliados == 0).sum()
)

print(
    "Coincidencia única:",
    (coincidencias_afiliados == 1).sum()
)

print(
    "Coincidencia múltiple:",
    (coincidencias_afiliados > 1).sum()
)

Sin coincidencia: 48304
Coincidencia única: 19432
Coincidencia múltiple: 77324


In [44]:
ids_no_en_afiliados = consumos.loc[
    coincidencias_afiliados == 0,
    "id_normalizado"
]

print(
    f"Compradores sin coincidencia directa en afiliados: "
    f"{len(ids_no_en_afiliados):,}"
)

Compradores sin coincidencia directa en afiliados: 48,304


In [45]:
conteo_persona_cargo = (
    grupo_familiar["id_persona_cargo_normalizado"]
    .value_counts()
)

coincidencias_cargo = (
    ids_no_en_afiliados
    .map(conteo_persona_cargo)
    .fillna(0)
    .astype(int)
)

resumen_cargo = (
    coincidencias_cargo
    .value_counts()
    .sort_index()
    .rename_axis("numero_coincidencias")
    .reset_index(name="compradores")
)

resumen_cargo["porcentaje"] = (
    resumen_cargo["compradores"]
    / len(ids_no_en_afiliados)
    * 100
).round(2)

display(resumen_cargo)

,numero_coincidencias,compradores,porcentaje
0,0,40946,84.77
1,1,2711,5.61
2,2,1914,3.96
3,3,1258,2.60
4,4,715,1.48
5,5,382,0.79
6,6,189,0.39
7,7,106,0.22
8,8,56,0.12
9,9,13,0.03


### 1.8 Validación de la ambigüedad de las dimensiones analíticas

La homologación de las llaves permitió identificar coincidencias entre los compradores de Piscilago y la base de afiliados. Sin embargo, una proporción importante de compradores presenta más de un registro asociado al mismo número de identificación.

Dado que los indicadores solicitados requieren segmentar los resultados por categoría, segmento poblacional y pirámide, se evalúa si las coincidencias múltiples conservan los mismos atributos analíticos o generan asignaciones ambiguas.

No se selecciona arbitrariamente un registro cuando existen múltiples coincidencias.

In [46]:
# IDs de compradores con más de una coincidencia en afiliados

ids_ambiguos = consumos.loc[
    coincidencias_afiliados > 1,
    "id_normalizado"
]

afiliados_compradores_ambiguos = afiliados[
    afiliados["id_normalizado"].isin(ids_ambiguos)
]

dimensiones = [
    "Categoria",
    "Segmento_poblacional",
    "Piramide1",
    "Piramide2"
]

resultado_ambiguedad = []

for dimension in dimensiones:

    valores_por_id = (
        afiliados_compradores_ambiguos
        .groupby("id_normalizado")[dimension]
        .nunique(dropna=False)
    )

    resultado_ambiguedad.append({
        "dimension": dimension,
        "ids_evaluados": len(valores_por_id),
        "ids_con_un_valor": (valores_por_id == 1).sum(),
        "ids_con_multiples_valores": (valores_por_id > 1).sum()
    })

resultado_ambiguedad = pd.DataFrame(resultado_ambiguedad)

resultado_ambiguedad["porcentaje_ambiguo"] = (
    resultado_ambiguedad["ids_con_multiples_valores"]
    / resultado_ambiguedad["ids_evaluados"]
    * 100
).round(2)

display(resultado_ambiguedad)

,dimension,ids_evaluados,ids_con_un_valor,ids_con_multiples_valores,porcentaje_ambiguo
0,Categoria,77324,49511,27813,35.97
1,Segmento_poblacional,77324,35212,42112,54.46
2,Piramide1,77324,24155,53169,68.76
3,Piramide2,77324,10887,66437,85.92


In [47]:
perfil_analitico = (
    afiliados_compradores_ambiguos
    .groupby("id_normalizado")[dimensiones]
    .nunique(dropna=False)
)

perfil_analitico["perfil_inequivoco"] = (
    perfil_analitico.eq(1).all(axis=1)
)

print(
    "Compradores con coincidencia múltiple:",
    f"{len(perfil_analitico):,}"
)

print(
    "Perfil analítico inequívoco:",
    f"{perfil_analitico['perfil_inequivoco'].sum():,}"
)

print(
    "Perfil analítico ambiguo:",
    f"{(~perfil_analitico['perfil_inequivoco']).sum():,}"
)

Compradores con coincidencia múltiple: 77,324
Perfil analítico inequívoco: 4,639
Perfil analítico ambiguo: 72,685


In [48]:
print(
    "Porcentaje con perfil inequívoco:",
    f"{perfil_analitico['perfil_inequivoco'].mean() * 100:.2f}%"
)

Porcentaje con perfil inequívoco: 6.00%


### 1.9 Clasificación de compradores según vínculo

La identificación de los compradores se realiza de forma jerárquica, siguiendo la lógica definida para el ejercicio:

1. Se verifica primero la existencia de la identificación en la base de afiliados.
2. Cuando no existe coincidencia directa, se consulta la base de personas a cargo.
3. Las identificaciones que no aparecen en ninguna de las dos fuentes se clasifican como no afiliadas.

La existencia de múltiples registros asociados a una identificación no modifica su condición de pertenencia; sin embargo, sí se conserva como una alerta de calidad para la posterior asignación de atributos de segmentación.

In [49]:
clasificacion = consumos[
    ["id_normalizado"]
].copy()

clasificacion["n_coincidencias_afiliado"] = (
    clasificacion["id_normalizado"]
    .map(conteo_afiliados_por_id)
    .fillna(0)
    .astype(int)
)

clasificacion["es_afiliado"] = (
    clasificacion["n_coincidencias_afiliado"] > 0
)

In [50]:
ids_personas_cargo = set(
    grupo_familiar["id_persona_cargo_normalizado"]
    .dropna()
)

clasificacion["es_persona_cargo"] = (
    clasificacion["id_normalizado"]
    .isin(ids_personas_cargo)
)

In [51]:
import numpy as np

clasificacion["tipo_vinculo"] = np.select(
    [
        clasificacion["es_afiliado"],
        (~clasificacion["es_afiliado"]) &
        clasificacion["es_persona_cargo"]
    ],
    [
        "Afiliado",
        "Grupo familiar"
    ],
    default="No afiliado"
)

In [52]:
resumen_vinculo = (
    clasificacion["tipo_vinculo"]
    .value_counts()
    .rename_axis("tipo_vinculo")
    .reset_index(name="compradores")
)

resumen_vinculo["porcentaje"] = (
    resumen_vinculo["compradores"]
    / len(clasificacion)
    * 100
).round(2)

display(resumen_vinculo)

print(
    "Total clasificado:",
    f"{resumen_vinculo['compradores'].sum():,}"
)

print(
    "Total consumos:",
    f"{len(consumos):,}"
)

,tipo_vinculo,compradores,porcentaje
0,Afiliado,96756,66.70
1,No afiliado,40946,28.23
2,Grupo familiar,7358,5.07


Total clasificado: 145,060
Total consumos: 145,060


In [53]:
ids_perfil_consistente = set(
    perfil_analitico[
        perfil_analitico["perfil_inequivoco"]
    ].index
)

In [54]:
clasificacion["calidad_segmentacion"] = np.select(
    [
        clasificacion["n_coincidencias_afiliado"] == 1,

        (clasificacion["n_coincidencias_afiliado"] > 1) &
        clasificacion["id_normalizado"].isin(ids_perfil_consistente),

        clasificacion["n_coincidencias_afiliado"] > 1
    ],
    [
        "Inequivoca",
        "Multiple consistente",
        "Ambigua"
    ],
    default="No aplica"
)

In [55]:
resumen_calidad = (
    clasificacion["calidad_segmentacion"]
    .value_counts()
    .rename_axis("calidad_segmentacion")
    .reset_index(name="compradores")
)

resumen_calidad["porcentaje"] = (
    resumen_calidad["compradores"]
    / len(clasificacion)
    * 100
).round(2)

display(resumen_calidad)

,calidad_segmentacion,compradores,porcentaje
0,Ambigua,72685,50.11
1,No aplica,48304,33.30
2,Inequivoca,19432,13.40
3,Multiple consistente,4639,3.20


#### Hallazgo de calidad de datos

La integración evidencia una limitación relevante en la identificación de afiliados. De los 77.324 compradores con múltiples coincidencias en `BD_AFILIADOS`, únicamente 4.639 (6,0 %) conservan el mismo perfil en categoría, segmento poblacional y pirámides. En los 72.685 restantes existe conflicto en al menos una de estas dimensiones.

Por esta razón, las coincidencias múltiples no se resuelven mediante selección arbitraria de registros. La pertenencia a la base de afiliados se conserva para la clasificación del comprador, mientras que la asignación de atributos de segmentación se controla mediante una bandera de calidad.

## 2. Construcción de la tabla analítica de Piscilago

A partir del perfilamiento y las validaciones de calidad, se construye una tabla analítica a nivel de comprador.

La integración conserva los 145.060 registros originales de consumo y aplica la siguiente jerarquía de clasificación:

1. **Afiliado:** la identificación del comprador aparece directamente en `BD_AFILIADOS`.
2. **Grupo familiar:** no aparece como afiliado, pero sí como persona a cargo en `AFILIADOS_A_CARGO`.
3. **No afiliado:** no presenta coincidencia en ninguna de las dos fuentes.

La asignación de atributos de afiliación se realiza únicamente cuando la información disponible permite una asociación consistente, evitando multiplicar registros o seleccionar arbitrariamente entre perfiles diferentes.

In [56]:
# Compradores clasificados como grupo familiar

ids_grupo_familiar = set(
    clasificacion.loc[
        clasificacion["tipo_vinculo"] == "Grupo familiar",
        "id_normalizado"
    ]
)

print(f"Compradores del grupo familiar: {len(ids_grupo_familiar):,}")

Compradores del grupo familiar: 7,358


In [57]:
relaciones_grupo_consumidores = (
    grupo_familiar[
        grupo_familiar["id_persona_cargo_normalizado"]
        .isin(ids_grupo_familiar)
    ][
        [
            "id_persona_cargo_normalizado",
            "id_afiliado_normalizado"
        ]
    ]
    .drop_duplicates()
)

print(
    "Relaciones únicas encontradas:",
    f"{len(relaciones_grupo_consumidores):,}"
)

Relaciones únicas encontradas: 17,677


In [58]:
titulares_por_persona_cargo = (
    relaciones_grupo_consumidores
    .groupby("id_persona_cargo_normalizado")
    ["id_afiliado_normalizado"]
    .nunique()
)

resumen_titulares = (
    titulares_por_persona_cargo
    .value_counts()
    .sort_index()
    .rename_axis("numero_titulares")
    .reset_index(name="personas_grupo_familiar")
)

resumen_titulares["porcentaje"] = (
    resumen_titulares["personas_grupo_familiar"]
    / len(ids_grupo_familiar)
    * 100
).round(2)

display(resumen_titulares)

,numero_titulares,personas_grupo_familiar,porcentaje
0,1,2711,36.84
1,2,1914,26.01
2,3,1258,17.10
3,4,715,9.72
4,5,382,5.19
5,6,189,2.57
6,7,106,1.44
7,8,56,0.76
8,9,13,0.18
9,10,8,0.11


In [59]:
ids_afiliados_disponibles = set(
    afiliados["id_normalizado"].dropna()
)

relaciones_grupo_consumidores["titular_en_bd_afiliados"] = (
    relaciones_grupo_consumidores["id_afiliado_normalizado"]
    .isin(ids_afiliados_disponibles)
)

resumen_titulares_bd = (
    relaciones_grupo_consumidores["titular_en_bd_afiliados"]
    .value_counts()
    .rename_axis("titular_en_bd_afiliados")
    .reset_index(name="relaciones")
)

display(resumen_titulares_bd)

,titular_en_bd_afiliados,relaciones
0,True,17346
1,False,331


In [60]:
titular_valido_por_persona = (
    relaciones_grupo_consumidores
    .groupby("id_persona_cargo_normalizado")
    ["titular_en_bd_afiliados"]
    .any()
)

print(
    "Personas con al menos un titular localizado:",
    f"{titular_valido_por_persona.sum():,}"
)

print(
    "Personas sin ningún titular localizado:",
    f"{(~titular_valido_por_persona).sum():,}"
)

Personas con al menos un titular localizado: 7,285
Personas sin ningún titular localizado: 73


In [61]:
conteo_perfiles_titular = (
    afiliados
    .groupby("id_normalizado")
    .size()
)

relaciones_grupo_consumidores["n_perfiles_titular"] = (
    relaciones_grupo_consumidores["id_afiliado_normalizado"]
    .map(conteo_perfiles_titular)
    .fillna(0)
    .astype(int)
)

resumen_perfiles_titular = (
    relaciones_grupo_consumidores["n_perfiles_titular"]
    .value_counts()
    .sort_index()
    .rename_axis("numero_perfiles_titular")
    .reset_index(name="relaciones")
)

display(resumen_perfiles_titular)

,numero_perfiles_titular,relaciones
0,0,331
1,1,1339
2,2,16007


### 2.1 Validación de atributos para grupo familiar

Las personas clasificadas como grupo familiar pueden estar asociadas a más de un afiliado titular y, adicionalmente, algunos titulares presentan más de un perfil en `BD_AFILIADOS`.

Para evitar asignaciones arbitrarias, se expanden temporalmente las relaciones con los posibles perfiles del titular y se verifica si las dimensiones analíticas requeridas por el ejercicio son consistentes.

La expansión se utiliza únicamente para evaluar consistencia; posteriormente se retorna a una única fila por comprador.

In [62]:
dimensiones = [
    "Categoria",
    "Segmento_poblacional",
    "Piramide1",
    "Piramide2"
]

perfiles_afiliados = afiliados[
    [
        "id_normalizado",
        "Categoria",
        "Segmento_poblacional",
        "Piramide1",
        "Piramide2"
    ]
].copy()

candidatos_grupo = relaciones_grupo_consumidores.merge(
    perfiles_afiliados,
    how="left",
    left_on="id_afiliado_normalizado",
    right_on="id_normalizado"
)

print(
    "Filas temporales después del cruce:",
    f"{len(candidatos_grupo):,}"
)

print(
    "Personas a cargo representadas:",
    f"{candidatos_grupo['id_persona_cargo_normalizado'].nunique():,}"
)

Filas temporales después del cruce: 33,684
Personas a cargo representadas: 7,358


In [63]:
consistencia_grupo = (
    candidatos_grupo
    .groupby("id_persona_cargo_normalizado")[dimensiones]
    .nunique(dropna=True)
)

display(consistencia_grupo.head())

,Categoria,Segmento_poblacional,Piramide1,Piramide2
id_persona_cargo_normalizado,,,,
10025908,0,0,0,0
10027318,1,1,2,2
10047996,1,1,1,1
10051343,1,1,1,1
10057185,1,1,2,2


In [64]:
consistencia_grupo["perfil_inequivoco"] = (
    consistencia_grupo.eq(1).all(axis=1)
)

In [65]:
ids_sin_titular = set(
    titular_valido_por_persona[
        ~titular_valido_por_persona
    ].index
)

In [66]:
print(
    "Personas grupo familiar evaluadas:",
    f"{len(consistencia_grupo):,}"
)

print(
    "Perfil analítico consistente:",
    f"{consistencia_grupo['perfil_inequivoco'].sum():,}"
)

print(
    "Perfil analítico ambiguo:",
    f"{(~consistencia_grupo['perfil_inequivoco']).sum():,}"
)

print(
    "Sin titular localizado:",
    f"{len(ids_sin_titular):,}"
)

Personas grupo familiar evaluadas: 7,358
Perfil analítico consistente: 385
Perfil analítico ambiguo: 6,973
Sin titular localizado: 73


In [67]:
calidad_grupo = (
    consistencia_grupo[
        ["perfil_inequivoco"]
    ]
    .reset_index()
)

calidad_grupo["calidad_segmentacion_grupo"] = np.select(
    [
        calidad_grupo["id_persona_cargo_normalizado"]
        .isin(ids_sin_titular),

        calidad_grupo["perfil_inequivoco"]
    ],
    [
        "Sin titular localizado",
        "Consistente"
    ],
    default="Ambigua"
)

In [68]:
resumen_calidad_grupo = (
    calidad_grupo["calidad_segmentacion_grupo"]
    .value_counts()
    .rename_axis("calidad_segmentacion_grupo")
    .reset_index(name="compradores")
)

resumen_calidad_grupo["porcentaje"] = (
    resumen_calidad_grupo["compradores"]
    / len(ids_grupo_familiar)
    * 100
).round(2)

display(resumen_calidad_grupo)

,calidad_segmentacion_grupo,compradores,porcentaje
0,Ambigua,6900,93.78
1,Consistente,385,5.23
2,Sin titular localizado,73,0.99


In [69]:
ids_grupo_consistentes = set(
    calidad_grupo.loc[
        calidad_grupo["calidad_segmentacion_grupo"] == "Consistente",
        "id_persona_cargo_normalizado"
    ]
)

In [70]:
atributos_grupo_consistente = (
    candidatos_grupo[
        candidatos_grupo["id_persona_cargo_normalizado"]
        .isin(ids_grupo_consistentes)
    ]
    .groupby("id_persona_cargo_normalizado")[dimensiones]
    .first()
    .reset_index()
)

In [71]:
print(
    "Personas con atributos recuperados:",
    f"{len(atributos_grupo_consistente):,}"
)

display(atributos_grupo_consistente.head())

Personas con atributos recuperados: 385


,id_persona_cargo_normalizado,Categoria,Segmento_poblacional,Piramide1,Piramide2
0,10047996,B,Medio,1 emp grandes,1.2 premium
1,10051343,A,Medio,4 micro,4.4 trans.natural ent. 11 a 99 trab.
2,10064289,A,Basico,4 micro,4.5 transaccional
3,10068254,A,Basico,4 micro,4.2 trans. mas de 100 trab.
4,10074388,A,Medio,1 emp grandes,1.1 platinum


In [72]:
# Dimensiones necesarias para el ejercicio
dimensiones = [
    "Categoria",
    "Segmento_poblacional",
    "Piramide1",
    "Piramide2"
]

# IDs de afiliados directos presentes en los consumos
ids_afiliados_consumidores = set(
    clasificacion.loc[
        clasificacion["tipo_vinculo"] == "Afiliado",
        "id_normalizado"
    ]
)

# Registros candidatos en la base de afiliados
candidatos_afiliados = afiliados[
    afiliados["id_normalizado"].isin(ids_afiliados_consumidores)
].copy()

# Consistencia de las dimensiones por comprador
consistencia_afiliados = (
    candidatos_afiliados
    .groupby("id_normalizado")[dimensiones]
    .nunique(dropna=True)
)

consistencia_afiliados["perfil_inequivoco"] = (
    consistencia_afiliados.eq(1).all(axis=1)
)

print(
    "Afiliados compradores:",
    f"{len(consistencia_afiliados):,}"
)

print(
    "Perfil consistente:",
    f"{consistencia_afiliados['perfil_inequivoco'].sum():,}"
)

print(
    "Perfil ambiguo:",
    f"{(~consistencia_afiliados['perfil_inequivoco']).sum():,}"
)

Afiliados compradores: 96,756
Perfil consistente: 24,071
Perfil ambiguo: 72,685


In [73]:
ids_afiliados_consistentes = set(
    consistencia_afiliados.loc[
        consistencia_afiliados["perfil_inequivoco"]
    ].index
)

atributos_afiliados_consistentes = (
    candidatos_afiliados[
        candidatos_afiliados["id_normalizado"]
        .isin(ids_afiliados_consistentes)
    ]
    .groupby("id_normalizado")[dimensiones]
    .first()
    .reset_index()
)

print(
    "Afiliados con atributos recuperados:",
    f"{len(atributos_afiliados_consistentes):,}"
)

Afiliados con atributos recuperados: 24,071


In [74]:
piscilago = consumos.copy()

print(f"Registros iniciales: {len(piscilago):,}")

Registros iniciales: 145,060


In [75]:
piscilago = piscilago.merge(
    clasificacion[
        ["id_normalizado", "tipo_vinculo"]
    ],
    how="left",
    on="id_normalizado",
    validate="one_to_one"
)

print(f"Registros después de clasificación: {len(piscilago):,}")
print(piscilago["tipo_vinculo"].value_counts(dropna=False))

Registros después de clasificación: 145,060
tipo_vinculo
Afiliado          96756
No afiliado       40946
Grupo familiar     7358
Name: count, dtype: int64


In [76]:
atributos_afiliados = (
    atributos_afiliados_consistentes
    .rename(
        columns={
            "Categoria": "Categoria_afiliado",
            "Segmento_poblacional": "Segmento_afiliado",
            "Piramide1": "Piramide1_afiliado",
            "Piramide2": "Piramide2_afiliado"
        }
    )
)

In [77]:
piscilago = piscilago.merge(
    atributos_afiliados,
    how="left",
    on="id_normalizado",
    validate="one_to_one"
)

print(f"Registros después de atributos afiliado: {len(piscilago):,}")

Registros después de atributos afiliado: 145,060


In [78]:
atributos_grupo = (
    atributos_grupo_consistente
    .rename(
        columns={
            "id_persona_cargo_normalizado": "id_normalizado",
            "Categoria": "Categoria_grupo",
            "Segmento_poblacional": "Segmento_grupo",
            "Piramide1": "Piramide1_grupo",
            "Piramide2": "Piramide2_grupo"
        }
    )
)

In [79]:
piscilago = piscilago.merge(
    atributos_grupo,
    how="left",
    on="id_normalizado",
    validate="one_to_one"
)

print(f"Registros después de atributos grupo familiar: {len(piscilago):,}")

Registros después de atributos grupo familiar: 145,060


In [80]:
piscilago["Categoria"] = (
    piscilago["Categoria_afiliado"]
    .combine_first(piscilago["Categoria_grupo"])
)

piscilago["Segmento_poblacional"] = (
    piscilago["Segmento_afiliado"]
    .combine_first(piscilago["Segmento_grupo"])
)

piscilago["Piramide1"] = (
    piscilago["Piramide1_afiliado"]
    .combine_first(piscilago["Piramide1_grupo"])
)

piscilago["Piramide2"] = (
    piscilago["Piramide2_afiliado"]
    .combine_first(piscilago["Piramide2_grupo"])
)

In [81]:
piscilago["calidad_segmentacion"] = np.select(
    [
        piscilago["tipo_vinculo"].eq("No afiliado"),

        piscilago["tipo_vinculo"].eq("Afiliado")
        & piscilago["id_normalizado"].isin(ids_afiliados_consistentes),

        piscilago["tipo_vinculo"].eq("Grupo familiar")
        & piscilago["id_normalizado"].isin(ids_grupo_consistentes),

        piscilago["tipo_vinculo"].eq("Grupo familiar")
        & piscilago["id_normalizado"].isin(ids_sin_titular)
    ],
    [
        "No aplica",
        "Consistente",
        "Consistente",
        "Sin titular localizado"
    ],
    default="Ambigua"
)

In [82]:
resumen_final_segmentacion = (
    piscilago
    .groupby(
        ["tipo_vinculo", "calidad_segmentacion"],
        dropna=False
    )
    .size()
    .reset_index(name="compradores")
)

display(resumen_final_segmentacion)

,tipo_vinculo,calidad_segmentacion,compradores
0,Afiliado,Ambigua,72685
1,Afiliado,Consistente,24071
2,Grupo familiar,Ambigua,6900
3,Grupo familiar,Consistente,385
4,Grupo familiar,Sin titular localizado,73
5,No afiliado,No aplica,40946


In [83]:
piscilago["VALOR_SEGURO"] = np.where(
    piscilago["SEGURO"].eq(1),
    1200,
    0
)

piscilago["VALOR_NETO"] = (
    piscilago["VALOR_SUBTOTAL"]
    - piscilago["VALOR_SEGURO"]
)

In [84]:
print(
    "Compras con seguro:",
    f"{piscilago['SEGURO'].eq(1).sum():,}"
)

print(
    "Valor total subtotal:",
    f"${piscilago['VALOR_SUBTOTAL'].sum():,.0f}"
)

print(
    "Valor total seguros descontados:",
    f"${piscilago['VALOR_SEGURO'].sum():,.0f}"
)

print(
    "Valor neto analítico:",
    f"${piscilago['VALOR_NETO'].sum():,.0f}"
)

display(
    piscilago[
        [
            "IDENTIFICACION",
            "DESC_PRODUCTO",
            "VALOR_SUBTOTAL",
            "SEGURO",
            "VALOR_SEGURO",
            "VALOR_NETO"
        ]
    ].head(10)
)

Compras con seguro: 35,305
Valor total subtotal: $8,586,771,563
Valor total seguros descontados: $42,366,000
Valor neto analítico: $8,544,405,563


,IDENTIFICACION,DESC_PRODUCTO,VALOR_SUBTOTAL,SEGURO,VALOR_SEGURO,VALOR_NETO
0,10000,ENTRADA PISCILAGO,60000.0,1,1200,58800.0
1,10001,ENTRADA PISCILAGO,60000.0,0,0,60000.0
2,10002,ENTRADA PISCILAGO,53000.0,0,0,53000.0
3,10003,PISCITOUR,93333.0,0,0,93333.0
4,10004,ENTRADA PISCILAGO,60000.0,0,0,60000.0
5,10005,PISCITOUR,102333.0,1,1200,101133.0
6,10006,ENTRADA PISCILAGO,60000.0,0,0,60000.0
7,10010,ENTRADA PISCILAGO,60000.0,0,0,60000.0
8,10011,PISCITOUR,102333.0,0,0,102333.0
9,10013,PISCITOUR,86333.0,1,1200,85133.0


In [85]:
print("CONTROL FINAL")
print("-" * 50)

print(
    "1. Filas:",
    len(piscilago),
    "OK" if len(piscilago) == len(consumos) else "REVISAR"
)

print(
    "2. IDs únicos:",
    piscilago["id_normalizado"].nunique(),
    "OK" if piscilago["id_normalizado"].nunique() == len(consumos) else "REVISAR"
)

print(
    "3. Clasificación completa:",
    piscilago["tipo_vinculo"].notna().sum(),
    "OK" if piscilago["tipo_vinculo"].notna().all() else "REVISAR"
)

CONTROL FINAL
--------------------------------------------------
1. Filas: 145060 OK
2. IDs únicos: 145060 OK
3. Clasificación completa: 145060 OK


In [86]:
columnas_finales = [
    "PERIODO",
    "ANIO",
    "ESTADO",
    "FECHA_COMPRA",
    "DESC_PRODUCTO",
    "VALOR_SUBTOTAL",
    "SEGURO",
    "VALOR_SEGURO",
    "VALOR_NETO",
    "IDENTIFICACION",
    "NOMBRES",
    "PUNTO_RECAUDO",
    "id_normalizado",
    "tipo_vinculo",
    "Categoria",
    "Segmento_poblacional",
    "Piramide1",
    "Piramide2",
    "calidad_segmentacion"
]

piscilago_analitica = piscilago[columnas_finales].copy()

print(
    f"Tabla analítica: "
    f"{piscilago_analitica.shape[0]:,} filas x "
    f"{piscilago_analitica.shape[1]} columnas"
)

display(piscilago_analitica.head())

Tabla analítica: 145,060 filas x 19 columnas


,PERIODO,ANIO,ESTADO,FECHA_COMPRA,DESC_PRODUCTO,VALOR_SUBTOTAL,SEGURO,VALOR_SEGURO,VALOR_NETO,IDENTIFICACION,NOMBRES,PUNTO_RECAUDO,id_normalizado,tipo_vinculo,Categoria,Segmento_poblacional,Piramide1,Piramide2,calidad_segmentacion
0,202407,2024,FACTURADA,17/07/2024,ENTRADA PISCILAGO,60000.0,1,1200,58800.0,10000,C,TAQUILLA PISCILAGO,10000,Afiliado,NaN,NaN,NaN,NaN,Ambigua
1,202402,2024,FACTURADA,24/02/2024,ENTRADA PISCILAGO,60000.0,0,0,60000.0,10001,C,TAQUILLA PISCILAGO,10001,Afiliado,A,Basico,4 micro,4.3 trans.juridica ent. 11 a 99 trab.,Consistente
2,202401,2024,FACTURADA,1/01/2024,ENTRADA PISCILAGO,53000.0,0,0,53000.0,10002,IN,TAQUILLA PISCILAGO,10002,No afiliado,NaN,NaN,NaN,NaN,No aplica
3,202401,2024,FACTURADA,27/01/2024,PISCITOUR,93333.0,0,0,93333.0,10003,C,TAQUILLA PISCILAGO,10003,Afiliado,A,Basico,4 micro,4.5 transaccional,Consistente
4,202401,2024,FACTURADA,10/01/2024,ENTRADA PISCILAGO,60000.0,0,0,60000.0,10004,C,TAQUILLA PISCILAGO,10004,No afiliado,NaN,NaN,NaN,NaN,No aplica


In [87]:
piscilago_analitica["FECHA_COMPRA"] = pd.to_datetime(
    piscilago_analitica["FECHA_COMPRA"],
    dayfirst=True,
    errors="coerce"
)

print(
    "Fechas no convertidas:",
    piscilago_analitica["FECHA_COMPRA"].isna().sum()
)

print(
    "Fecha mínima:",
    piscilago_analitica["FECHA_COMPRA"].min()
)

print(
    "Fecha máxima:",
    piscilago_analitica["FECHA_COMPRA"].max()
)

Fechas no convertidas: 0
Fecha mínima: 2024-01-01 00:00:00
Fecha máxima: 2024-09-15 00:00:00


In [88]:
piscilago_analitica["MES"] = (
    piscilago_analitica["FECHA_COMPRA"].dt.month
)

piscilago_analitica["DIA_SEMANA"] = (
    piscilago_analitica["FECHA_COMPRA"].dt.day_name()
)

In [89]:
ruta_salida = DATA_PROCESSED / "piscilago_analitica.csv"

piscilago_analitica.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo generado:")
print(ruta_salida)

Archivo generado:
c:\Users\acoba\Documents\GitHub\Prueba_Analista_BI\data\processed\piscilago_analitica.csv


## 3. Construcción de la dimensión poblacional de afiliados

Para calcular indicadores de penetración se requiere un universo poblacional de referencia independiente de los compradores de Piscilago.

La fuente `BD_AFILIADOS` presenta múltiples registros asociados a un mismo número de identificación y, en algunos casos, perfiles analíticos diferentes. Por esta razón, la dimensión poblacional se construye únicamente con identificadores cuya clasificación sea consistente en las dimensiones requeridas para el ejercicio:

- Categoría
- Segmento poblacional
- Pirámide 1
- Pirámide 2

La dimensión resultante contiene una única fila por identificador analíticamente consistente. Los casos ambiguos se conservan como hallazgo de calidad, pero no se asignan arbitrariamente a una categoría o segmento.

In [90]:
dimensiones_poblacion = [
    "Categoria",
    "Segmento_poblacional",
    "Piramide1",
    "Piramide2"
]

consistencia_poblacion = (
    afiliados
    .groupby("id_normalizado")[dimensiones_poblacion]
    .nunique(dropna=False)
)

consistencia_poblacion["perfil_consistente"] = (
    consistencia_poblacion.eq(1).all(axis=1)
)

print(
    "Identificadores únicos evaluados:",
    f"{len(consistencia_poblacion):,}"
)

print(
    "Perfil consistente:",
    f"{consistencia_poblacion['perfil_consistente'].sum():,}"
)

print(
    "Perfil ambiguo:",
    f"{(~consistencia_poblacion['perfil_consistente']).sum():,}"
)

Identificadores únicos evaluados: 303,956
Perfil consistente: 168,866
Perfil ambiguo: 135,090


In [91]:
resumen_calidad_poblacion = pd.DataFrame({
    "calidad": ["Consistente", "Ambigua"],
    "identificadores": [
        consistencia_poblacion["perfil_consistente"].sum(),
        (~consistencia_poblacion["perfil_consistente"]).sum()
    ]
})

resumen_calidad_poblacion["porcentaje"] = (
    resumen_calidad_poblacion["identificadores"]
    / len(consistencia_poblacion)
    * 100
).round(2)

display(resumen_calidad_poblacion)

,calidad,identificadores,porcentaje
0,Consistente,168866,55.56
1,Ambigua,135090,44.44


In [92]:
ids_poblacion_consistentes = set(
    consistencia_poblacion.loc[
        consistencia_poblacion["perfil_consistente"]
    ].index
)

print(
    "IDs consistentes:",
    f"{len(ids_poblacion_consistentes):,}"
)

IDs consistentes: 168,866


In [93]:
afiliados_consistentes = afiliados[
    afiliados["id_normalizado"].isin(
        ids_poblacion_consistentes
    )
].copy()

In [94]:
dim_afiliados_consistentes = (
    afiliados_consistentes
    .groupby("id_normalizado")
    .agg(
        Categoria=("Categoria", "first"),
        Segmento_poblacional=("Segmento_poblacional", "first"),
        Piramide1=("Piramide1", "first"),
        Piramide2=("Piramide2", "first"),
        registros_fuente=("id_normalizado", "size")
    )
    .reset_index()
)

In [95]:
dim_afiliados_consistentes["calidad_perfil"] = "Consistente"

In [96]:
display(dim_afiliados_consistentes.head())

print(
    f"Dimensión poblacional: "
    f"{dim_afiliados_consistentes.shape[0]:,} filas x "
    f"{dim_afiliados_consistentes.shape[1]} columnas"
)

,id_normalizado,Categoria,Segmento_poblacional,Piramide1,Piramide2,registros_fuente,calidad_perfil
0,10,A,Medio,4 micro,4.5 transaccional,1,Consistente
1,10000000,A,Basico,2 emp medio,2.2 silver,1,Consistente
2,10000004,A,Basico,1 emp grandes,1.1 platinum,2,Consistente
3,10000009,A,Basico,4 micro,4.5 transaccional,2,Consistente
4,10000041,A,Basico,4 micro,4.5 transaccional,2,Consistente


Dimensión poblacional: 168,866 filas x 7 columnas


In [97]:
print(
    "IDs únicos:",
    dim_afiliados_consistentes["id_normalizado"].nunique()
)

print(
    "Filas:",
    len(dim_afiliados_consistentes)
)

print(
    "¿La llave es única?:",
    dim_afiliados_consistentes["id_normalizado"].is_unique
)

IDs únicos: 168866
Filas: 168866
¿La llave es única?: True


In [98]:
display(
    dim_afiliados_consistentes[
        dimensiones_poblacion
    ]
    .isna()
    .sum()
)

Categoria               0
Segmento_poblacional    0
Piramide1               0
Piramide2               0
dtype: int64

In [99]:
for columna in dimensiones_poblacion:
    print("\n", columna)
    print(
        dim_afiliados_consistentes[columna]
        .value_counts()
        .sort_index()
    )


 Categoria
Categoria
A    135621
B     15497
C     17748
Name: count, dtype: int64

 Segmento_poblacional
Segmento_poblacional
Alto       1769
Basico    96235
Joven     29086
Medio     41776
Name: count, dtype: int64

 Piramide1
Piramide1
1 emp grandes       35993
2 emp medio         22253
3 empresas pymes    25537
4 micro             84186
5 micro               897
Name: count, dtype: int64

 Piramide2
Piramide2
1.1 platinum                             24300
1.2 premium                              11693
2.1 gold                                 12493
2.2 silver                                9760
3.1 vip                                  13219
3.2 vip estandar                         12318
4.1 estandar                             14331
4.2 trans. mas de 100 trab.               4456
4.3 trans.juridica ent. 11 a 99 trab.    11783
4.4 trans.natural ent. 11 a 99 trab.       293
4.5 transaccional                        47542
4.6 transaccional - facultativo             41
4.7 transaccional 

In [102]:
poblacion_categoria = (
    dim_afiliados_consistentes
    .groupby("Categoria")
    ["id_normalizado"]
    .nunique()
    .reset_index(name="total_afiliados")
)

display(poblacion_categoria)

,Categoria,total_afiliados
0,A,135621
1,B,15497
2,C,17748


In [103]:
poblacion_segmento = (
    dim_afiliados_consistentes
    .groupby("Segmento_poblacional")
    ["id_normalizado"]
    .nunique()
    .reset_index(name="total_afiliados")
)

display(poblacion_segmento)

,Segmento_poblacional,total_afiliados
0,Alto,1769
1,Basico,96235
2,Joven,29086
3,Medio,41776


In [104]:
poblacion_piramide1 = (
    dim_afiliados_consistentes
    .groupby("Piramide1")
    ["id_normalizado"]
    .nunique()
    .reset_index(name="total_afiliados")
)

display(poblacion_piramide1)

,Piramide1,total_afiliados
0,1 emp grandes,35993
1,2 emp medio,22253
2,3 empresas pymes,25537
3,4 micro,84186
4,5 micro,897


In [105]:
poblacion_piramide2 = (
    dim_afiliados_consistentes
    .groupby("Piramide2")
    ["id_normalizado"]
    .nunique()
    .reset_index(name="total_afiliados")
)

display(poblacion_piramide2)

,Piramide2,total_afiliados
0,1.1 platinum,24300
1,1.2 premium,11693
2,2.1 gold,12493
3,2.2 silver,9760
4,3.1 vip,13219
5,3.2 vip estandar,12318
6,4.1 estandar,14331
7,4.2 trans. mas de 100 trab.,4456
8,4.3 trans.juridica ent. 11 a 99 trab.,11783
9,4.4 trans.natural ent. 11 a 99 trab.,293


In [106]:
ruta_dim_afiliados = (
    DATA_PROCESSED
    / "dim_afiliados_consistentes.csv"
)

dim_afiliados_consistentes.to_csv(
    ruta_dim_afiliados,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo generado:")
print(ruta_dim_afiliados)

Archivo generado:
c:\Users\acoba\Documents\GitHub\Prueba_Analista_BI\data\processed\dim_afiliados_consistentes.csv
